# Fine-Tuned Evaluation on Google Colab

**Model:** `Qwen/Qwen3-VL-4B-Instruct` + the trained LoRA adapter (PEFT).

> The baseline and the trained adapter must both be inside the project ZIP.
> Section 3 refuses to continue otherwise.

| Setting | Value |
|---|---|
| Evaluation set | the **same** 178 frozen held-out images |
| Prompts | the **same** 3 frozen prompts |
| Generations | 178 x 3 = 534 |
| Decoding | greedy (`do_sample=False`) |
| Seed | 42 |
| Precision | fp16 on a T4 (identical to the baseline) |
| Quantization | none — the baseline was not quantized either |

> **Runtime -> Change runtime type -> T4 GPU** before running.

Every setting above is asserted against `results/baseline/run_metadata.json`
at the end of section 4. If any of them drifted, the cell fails rather than
producing a comparison that looks valid but is not.

The defect classes in this dataset are **synthetic**. Results describe
synthetic-defect detection, not real industrial defect recognition.


## 1. Setup — extract the project from Drive

In [ ]:
# ============================================================
#  ONE-CELL SETUP — safe to re-run after any kernel restart
# ============================================================
import os, sys, time, shutil, zipfile, json
from pathlib import Path

import torch
assert torch.cuda.is_available(), 'No CUDA GPU. Runtime -> Change runtime type -> T4 GPU.'
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_MEM_GB = props.name, round(props.total_memory / 1e9, 1)
print(f'GPU        : {GPU_NAME} | {GPU_MEM_GB} GB | compute {props.major}.{props.minor}')
print(f'native bf16: {props.major >= 8} (False on T4 -> fp16)')

IN_COLAB = 'google.colab' in sys.modules
PROJECT = Path('/content/casting-defect-vlm')

def _looks_like_project(d):
    return (d / 'config' / 'config.yaml').exists() and (d / 'src' / 'prepare_data.py').exists()

if not _looks_like_project(PROJECT):
    found = [d for d in Path('/content').iterdir() if d.is_dir() and _looks_like_project(d)]
    if found:
        PROJECT = found[0]
    elif IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        MARKERS = ['config/config.yaml', 'src/prompts.py',
                   'scripts/run_baseline.py', 'results/baseline/eval_subset.json']
        best = None
        for zp in sorted(Path('/content/drive/MyDrive').glob('*.zip')):
            try:
                with zipfile.ZipFile(zp) as zf:
                    names = zf.namelist()
            except Exception:
                continue
            tops = {n.split('/')[0] for n in names if '/' in n}
            pre = (next(iter(tops)) + '/') if len(tops) == 1 else ''
            stripped = {n[len(pre):] if pre and n.startswith(pre) else n for n in names}
            if all(m in stripped for m in MARKERS):
                best = zp; break
        assert best, 'No project ZIP found in MyDrive.'
        print(f'extracting {best.name} ...')
        with zipfile.ZipFile(best) as zf:
            zf.extractall('/content')
        cands = [d for d in Path('/content').iterdir() if d.is_dir() and _looks_like_project(d)]
        assert cands, 'extracted but project folder not found'
        PROJECT = cands[0]
    else:
        raise SystemExit('project not found')
else:
    print('project already extracted — reusing')

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('PROJECT    :', PROJECT)

# portable image paths (idempotent)
target = PROJECT / 'src' / 'prepare_data.py'
MARKER = '# --- portability hotfix ---'
_src = target.read_text()
if MARKER not in _src:
    target.write_text(_src + '\n\n' + MARKER + '''
def load_jsonl(path, image_root=None, repair_image_paths=True):
    """Load JSONL, rebuilding absolute image paths for THIS machine."""
    import json as _json
    from .data_analysis import find_dataset_root
    from .utils import load_config, resolve_path
    p = resolve_path(path)
    if not p.exists():
        raise FileNotFoundError(f"{p} not found.")
    with p.open("r", encoding="utf-8") as fh:
        examples = [_json.loads(line) for line in fh if line.strip()]
    if not repair_image_paths:
        return examples
    if image_root is None:
        image_root = find_dataset_root(load_config()["data"]["raw_dir"])
    root = resolve_path(image_root)
    for ex in examples:
        rel = ex.get("relpath")
        if not rel:
            continue
        rebuilt = str(root / rel)
        ex["image"] = rebuilt
        for msg in ex.get("messages", []):
            for part in msg.get("content", []):
                if isinstance(part, dict) and part.get("type") == "image":
                    part["image"] = rebuilt
    return examples
''')
    print('patched    : src/prepare_data.py')
else:
    print('patched    : already applied')

for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.') or m.startswith('scripts')]:
    del sys.modules[_m]

from src.utils import load_config
from src.prepare_data import load_jsonl
cfg = load_config()
_tr = load_jsonl(cfg['data']['train_file'])
_va = load_jsonl(cfg['data']['validation_file'])
_missing = [e for e in _tr + _va if not Path(e['image']).exists()]
print(f'train      : {len(_tr)} | validation: {len(_va)} | missing files: {len(_missing)}')
assert not _missing, f'missing images, e.g. {_missing[:2]}'
print('SETUP OK')

## 2. Dependencies

Colab ships `torchao 0.10.0`, which PEFT 0.20 rejects when loading a saved
adapter (`only versions above 0.16.0 are supported`). Nothing here uses
torchao, so it is removed rather than upgraded — upgrading it can drag in a
different torch, and the baseline ran on the torch Colab already has.


In [ ]:
!pip install -q 'transformers>=5.0.0' 'accelerate>=1.0.0' 'peft>=0.14.0' 'pyyaml>=6.0' 'pandas>=2.0' 'pillow>=10.0' 'matplotlib>=3.8' 'scipy>=1.11' 'scikit-learn>=1.4'
!pip uninstall -y -q torchao

import importlib, sys
importlib.invalidate_caches()
for _m in [k for k in list(sys.modules) if k.startswith(('peft', 'torchao'))]:
    del sys.modules[_m]

import transformers, peft, torch
from peft.import_utils import is_torchao_available
print('transformers      :', transformers.__version__)
print('peft              :', peft.__version__)
print('torch             :', torch.__version__)
print('torchao for peft  :', is_torchao_available(), '(must be False)')
print('VRAM allocated    :', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB (must be ~0)')

## 3. Evaluate the fine-tuned model

One cell: guards, the frozen inputs, the model with the adapter attached, 534
generations, metrics, the comparability check against the baseline, and the
results ZIP.

> **Keep this tab active.** If the session drops mid-run the partial results are
> lost with `/content` and the whole cell has to be re-run.

If this cell fails, stop and report the exact error. Do not change the model,
the prompts, the evaluation set or the decoding settings to get past it — that
would silently break the comparison this whole project exists to make.


In [ ]:
# =====================================================================
#  FINE-TUNED EVALUATION — same frozen 178 images, same 3 prompts,
#  same seed and greedy decoding as the baseline. ~50 min on a T4.
#  Loads the BASE model + the trained LoRA adapter (no 4-bit, so the
#  inference conditions match the baseline exactly).
# =====================================================================
import os, sys, gc, json, time, hashlib, shutil
from pathlib import Path

import torch
gc.collect(); torch.cuda.empty_cache()

props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_MEM_GB = props.name, round(props.total_memory / 1e9, 1)
IN_COLAB = 'google.colab' in sys.modules

PROJECT = Path('/content/casting-defect-vlm')
assert (PROJECT / 'config' / 'config.yaml').exists(), \
    'project not at /content/casting-defect-vlm — re-run the setup cell'
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.utils import load_config, resolve_path, set_seed
from src.prepare_data import load_jsonl
from src.baseline import select_eval_subset, build_balanced_subset, run_inference
from src.evaluate import evaluate_results_file
from src.inference import load_model
from src.prompts import EVAL_PROMPTS
from scripts.run_baseline import print_two_track_summary

cfg = load_config()
set_seed(cfg['project']['seed'])

# ---- guards -------------------------------------------------------
ADAPTER = resolve_path(cfg['training']['output_dir'])
assert (ADAPTER / 'adapter_config.json').exists(), f'no adapter at {ADAPTER}'
_ac = json.loads((ADAPTER / 'adapter_config.json').read_text())
assert _ac['base_model_name_or_path'] == cfg['model']['model_name'], \
    f"adapter was trained on {_ac['base_model_name_or_path']}"

import pandas as pd
_b = pd.read_csv(PROJECT / 'results/baseline/baseline_results.csv')
assert len(_b) == 534, f'baseline incomplete: {len(_b)} rows'
print(f'baseline  : {len(_b)} rows OK')
print(f'adapter   : {ADAPTER}  (r={_ac["r"]}, alpha={_ac["lora_alpha"]})')

# ---- the identical frozen inputs ----------------------------------
test_examples = load_jsonl(cfg['data']['test_file'])
subset = select_eval_subset(test_examples, n=None,
                            seed=cfg['project']['seed'], reuse=True)
balanced = build_balanced_subset(subset, seed=cfg['project']['seed'], reuse=True) \
    if cfg['baseline'].get('balanced_subset', True) else None
_sub = json.loads((PROJECT / 'results/baseline/eval_subset.json').read_text())
assert {e['relpath'] for e in subset} == set(_sub['relpaths']), \
    'evaluation set differs from the frozen baseline subset'
print(f'frozen set: {len(subset)} images (identical to baseline)')

# dtype is chosen exactly as the baseline chose it: fp16 on a T4.
use_bf16 = props.major >= 8
DTYPE = 'bfloat16' if use_bf16 else 'float16'
print(f'dtype     : {DTYPE} (native bf16: {use_bf16})')

t0 = time.time()
loaded = load_model(
    model_name=cfg['model']['model_name'],
    adapter_path=ADAPTER,                 # FINE-TUNED — adapter attached
    dtype=DTYPE,
    attn_implementation=cfg['model']['attn_implementation'],
    min_pixels=cfg['model'].get('min_pixels'),
    max_pixels=cfg['model'].get('max_pixels'),
)
MODEL_LOAD_SECONDS = time.time() - t0
assert loaded.adapter_path is not None, 'EVALUATION MUST USE THE FINE-TUNED MODEL'
print(f'model load: {MODEL_LOAD_SECONDS:.0f} s | VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

print(f'\nstarting {len(subset) * len(cfg["baseline"]["prompt_ids"])} generations '
      '— this takes ~50 min. Keep this tab active.\n')
t0 = time.time()
res_df = run_inference(
    loaded, subset,
    prompt_ids=cfg['baseline']['prompt_ids'],          # identical prompts
    max_new_tokens=cfg['model']['max_new_tokens'],
    do_sample=cfg['inference']['do_sample'],           # greedy
    results_csv=cfg['evaluation']['results_csv'],
    raw_jsonl=cfg['evaluation']['raw_outputs'],
    tag='finetuned',
)
TOTAL = time.time() - t0

payload = evaluate_results_file(
    cfg['evaluation']['results_csv'], 'results/evaluation', tag='finetuned',
    balanced_subset='results/baseline/balanced_subset.json' if balanced else None,
)
print_two_track_summary(payload, cfg, len(subset), len(res_df),
                        title='FINE-TUNED RESULTS (LoRA adapter attached)')

# ---- run metadata, mirroring the baseline's ------------------------
meta = {
    'run': 'finetuned',
    'fine_tuned': True,
    'adapter': str(ADAPTER),
    'lora': {k: _ac[k] for k in ('r', 'lora_alpha', 'lora_dropout', 'target_modules')},
    'model_id': cfg['model']['model_name'],
    'transformers_version': __import__('transformers').__version__,
    'torch_version': torch.__version__,
    'device': 'cuda:0',
    'dtype': f'torch.{DTYPE}',
    'attn_implementation': cfg['model']['attn_implementation'],
    'generation': {'do_sample': cfg['inference']['do_sample'],
                   'max_new_tokens': cfg['model']['max_new_tokens'],
                   'decoding': 'greedy'},
    'min_pixels': cfg['model'].get('min_pixels'),
    'max_pixels': cfg['model'].get('max_pixels'),
    'seed': cfg['project']['seed'],
    'prompt_ids': cfg['baseline']['prompt_ids'],
    'prompt_sha256_16': {p: hashlib.sha256(EVAL_PROMPTS[p].encode()).hexdigest()[:16]
                         for p in cfg['baseline']['prompt_ids']},
    'n_images': len(subset),
    'n_prompts': len(cfg['baseline']['prompt_ids']),
    'total_generations': len(res_df),
    'gpu_name': GPU_NAME,
    'gpu_memory_gb': GPU_MEM_GB,
    'model_load_seconds': round(MODEL_LOAD_SECONDS, 2),
    'total_inference_seconds': round(TOTAL, 2),
}
(PROJECT / 'results/evaluation/run_metadata.json').write_text(json.dumps(meta, indent=2))

# ---- validation: identical to the baseline's checks ----------------
expected = len(_sub['relpaths']) * len(cfg['baseline']['prompt_ids'])
pairs = set(zip(res_df['relpath'], res_df['prompt_id']))
print('\n' + '=' * 62)
print('VALIDATION')
print('=' * 62)
print(f'rows                : {len(res_df)} / {expected}')
print(f'unique image-prompt : {len(pairs)}')
print(f'evaluated set match : {set(res_df["relpath"]) == set(_sub["relpaths"])}')
print(f'generation errors   : {res_df["generation_error"].notna().sum()}')
print(f'unparseable         : {(res_df.parsed_prediction == "Unparseable").sum()}')
print(f'total wall time     : {TOTAL/60:.1f} min')
assert len(res_df) == expected, 'INCOMPLETE RUN'
assert len(pairs) == expected, 'duplicate/missing pairs'
assert set(res_df['relpath']) == set(_sub['relpaths']), 'evaluated images differ'

_bm = json.loads((PROJECT / 'results/baseline/run_metadata.json').read_text())
same = {k: (_bm.get(k) == meta.get(k)) for k in
        ('model_id', 'seed', 'prompt_ids', 'prompt_sha256_16', 'dtype',
         'min_pixels', 'max_pixels', 'n_images', 'total_generations')}
same['decoding'] = _bm['generation'] == meta['generation']
print('\ncomparability against the baseline run:')
for k, v in same.items():
    print(f'  {"OK  " if v else "DIFF"} {k}')
assert all(same.values()), 'settings drifted from the baseline — comparison invalid'
print('\nFINE-TUNED EVALUATION VALID')
print('=' * 62)

RES = PROJECT / 'results' / 'evaluation'
for f in sorted(RES.iterdir()):
    if f.is_file():
        print(f'  {f.name:<50}{f.stat().st_size:>12,} bytes')
arch = shutil.make_archive('/content/evaluation_results', 'zip', root_dir=str(RES))
print(f'\nwrote {arch} ({Path(arch).stat().st_size/1e6:.1f} MB)')

if IN_COLAB:
    from google.colab import files
    try:
        files.download('/content/evaluation_results.zip')
        print('download started')
    except Exception as e:
        print('download blocked:', e, '-> use the file browser (folder icon)')
print('\nDONE. Fine-tuned evaluation complete and packaged.')

## 4. STOP HERE

The fine-tuned evaluation is done. **Do not run the comparison in this
notebook** — it needs both runs side by side and belongs on your machine.

1. Confirm `evaluation_results.zip` downloaded.
2. Unzip it into the local project, then run the comparison:

```bash
cd ~/casting-defect-vlm
unzip -o ~/Downloads/evaluation_results.zip -d results/evaluation/
.venv/bin/python scripts/compare_models.py
```

Confidence values are model-reported, not calibrated probabilities. Track B
measures recognition of synthetic defect textures, not real industrial defect
recognition.
